# Email Categorization: Experiments
Pipeline: dataset → cleaning → tokenization → stop-word removal → TF-IDF → split → Logistic Regression & Naive Bayes → metrics.

Run this notebook from inside the `notebook/` folder (it adds the project root to the path).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from preprocess import clean_text

## 1. Load the dataset

In [ ]:
df = pd.read_csv('../dataset/emails.csv')
print(df.shape)
df.head(10)

In [ ]:
df['category'].value_counts().plot(kind='bar', color=['#dc2626', '#2563eb', '#16a34a'], title='Emails per category')
plt.ylabel('Count')
plt.show()

## 2. Text preprocessing
Lowercase → remove URLs/emails → remove special characters → tokenize → remove stop words.

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text', 'category']].head(8)

In [ ]:
example = 'WIN a FREE prize!!! Visit https://claim-now.example.com or email me@example.com'
print('Original:', example)
print('Cleaned :', clean_text(example))

## 3. Train/test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['category'], test_size=0.25, random_state=42, stratify=df['category'])
print(len(X_train), 'train /', len(X_test), 'test')

## 4. TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print('Vocabulary size:', len(vectorizer.vocabulary_))
print('Matrix shape  :', X_train_tfidf.shape)

## 5. Train and compare two models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=10, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.5),
}
LABELS = ['Personal', 'Spam', 'Work']
scores = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    scores[name] = accuracy_score(y_test, preds)
    print(name, '- accuracy:', round(scores[name], 3))
    print(classification_report(y_test, preds, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_test, preds, labels=LABELS), display_labels=LABELS).plot(cmap='Blues')
    plt.title(name)
    plt.show()

In [ ]:
pd.Series(scores).plot(kind='bar', ylim=(0, 1), title='Model accuracy comparison')
plt.ylabel('Accuracy')
plt.show()

## 6. Most important words per class (Logistic Regression)

In [ ]:
lr = models['Logistic Regression']
features = np.array(vectorizer.get_feature_names_out())
for i, label in enumerate(lr.classes_):
    top = np.argsort(lr.coef_[i])[-8:][::-1]
    print(f'{label:9s}:', ', '.join(features[top]))

## 7. Try your own email

In [ ]:
def classify(text):
    probs = lr.predict_proba(vectorizer.transform([clean_text(text)]))[0]
    return {c: round(float(p), 3) for c, p in zip(lr.classes_, probs)}

classify('Please send the quarterly report to the manager before the meeting')